In [33]:
"""
Compute Best‑Level Order‑Flow Imbalance (OFI) for a single stock.

Spec matches Section 2.1 of the project PDF (m = 1 bid / ask rules).

Author: ChatGPT “OFI Builder”
"""

from pathlib import Path
import pandas as pd


# ── CONFIGURE INPUT / OUTPUT PATHS ─────────────────────────────────────────────
LOB_CSV   = Path("first_25000_rows.csv")
OUT_CSV   = Path("best_level_ofi.csv")

# Column names in the LOB file
COL_TIME  = "ts_event"   # timestamp column
COL_BPX   = "bid_px_00"  # best bid price
COL_BSZ   = "bid_sz_00"  # best bid size
COL_APX   = "ask_px_00"  # best ask price
COL_ASZ   = "ask_sz_00"  # best ask size

# Resampling frequency (change to "30s", "5min", etc. if desired)
BUCKET    = "1min"


def load_lob(path: Path) -> pd.DataFrame:
    """Read the raw limit‑order‑book CSV (best level only)."""
    df = pd.read_csv(
        path,
        usecols=[COL_TIME, COL_BPX, COL_BSZ, COL_APX, COL_ASZ],
        dtype={COL_BPX: "float64", COL_BSZ: "int64",
               COL_APX: "float64", COL_ASZ: "int64"},
    )
    df[COL_TIME] = pd.to_datetime(df[COL_TIME], utc=True, errors="coerce")
    df = df.sort_values(COL_TIME, ignore_index=True)
    return df
    

def compute_best_level_ofi(df: pd.DataFrame) -> pd.Series:
    """
    Apply three‑case rules to every book update and resample into
    a time‑series of Best‑Level OFI.
    """
    # Previous‑state columns
    prev = df.shift()

    # Bid flow (m = 1)
    bid_flow = (
        (df[COL_BPX] > prev[COL_BPX]) * df[COL_BSZ] +
        (df[COL_BPX] == prev[COL_BPX]) * (df[COL_BSZ] - prev[COL_BSZ]) +
        (df[COL_BPX] < prev[COL_BPX]) * (-prev[COL_BSZ])
    )

    # Ask flow (m = 1)
    ask_flow = (
        (df[COL_APX] > prev[COL_APX]) * (-df[COL_ASZ]) +
        (df[COL_APX] == prev[COL_APX]) * (df[COL_ASZ] - prev[COL_ASZ]) +
        (df[COL_APX] < prev[COL_APX]) * (df[COL_ASZ])
    )

    # Event‑level OFI, then bucket into fixed windows
    ofi_event = bid_flow - ask_flow
    ofi_series = (
        ofi_event
        .rename("ofi_event")
        .to_frame()
        .set_index(df[COL_TIME])
        .resample(BUCKET, label="right")
        .sum(min_count=1)             # NaN if bucket has no events
        .rename(columns={"ofi_event": "best_level_ofi"})
        .squeeze()                    # return as Series, not DataFrame
    )
    return ofi_series


def main() -> None:
    df  = load_lob(LOB_CSV)
    ser = compute_best_level_ofi(df)

    # Save & echo a small preview
    ser.to_csv(OUT_CSV, header=True)
    print(f"Best‑Level OFI written to: {OUT_CSV}\n")
    print(ser.head(10).to_string())


if __name__ == "__main__":
    main()


Best‑Level OFI written to: best_level_ofi.csv

ts_event
2024-10-21 11:55:00+00:00   -394.0
2024-10-21 11:56:00+00:00   -814.0
2024-10-21 11:57:00+00:00    300.0
2024-10-21 11:58:00+00:00    779.0
2024-10-21 11:59:00+00:00    284.0
2024-10-21 12:00:00+00:00    350.0
2024-10-21 12:01:00+00:00    629.0
2024-10-21 12:02:00+00:00   -398.0
2024-10-21 12:03:00+00:00   -501.0
2024-10-21 12:04:00+00:00   -449.0
Freq: min
